# ISTAT Road Traffic Analysis

This project analyses road accidents across Italian municipalities between 2020 and 2024.

Road accident data was obtained from the official ISTAT SDMX API and stored locally in the raw data folder.

The analysis will combine road accident data with municipality population and surface area data from SITUAS.

In [1]:
import pandas as pd
import requests
from pathlib import Path

In [2]:
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"

headers = {
    "Accept": "application/vnd.sdmx.data+csv;version=1.0.0"
}

params = {
    "startPeriod": "2020",
    "endPeriod": "2025"
}

raw_path = Path("data/raw/istat_traffic_2020_2025.csv")

refresh = False

if raw_path.exists() and not refresh:
    print("Raw file already exists. Using local file.")

else:
    print("Downloading data from ISTAT API...")

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=(30, 900)
    )

    response.raise_for_status()

    raw_path.write_bytes(response.content)

    print("Status code:", response.status_code)
    print("Size:", len(response.content))
    print("Saved to:", raw_path)

Raw file already exists. Using local file.


In [3]:
df_raw = pd.read_csv(
    raw_path,
    encoding="utf-8",
    dtype={"REF_AREA": "string"}
)

df_raw.shape

(113856, 16)

In [4]:
df_raw["TIME_PERIOD"].value_counts().sort_index()

TIME_PERIOD
2020    23712
2021    23712
2022    23712
2023    23703
2024    19017
Name: count, dtype: int64

### Data availability

The ISTAT API request covered the period from 2020 to 2025.

However, the returned dataset contained observations only from 2020 to 2024.
No municipality-level observations were returned for 2025.

Therefore, the analysis uses the latest available period returned by this
dataset: 2020–2024.

## l'Initial Data Inspection

In [5]:
df_raw.head(20)

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,001001,KILLINJ,F,2020,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,001001,KILLINJ,F,2021,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,001001,KILLINJ,F,2022,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,001001,KILLINJ,F,2023,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,001001,KILLINJ,F,2024,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,IT1:41_983(1.0),A,001001,KILLINJ,M,2020,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,IT1:41_983(1.0),A,001001,KILLINJ,M,2021,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,IT1:41_983(1.0),A,001001,KILLINJ,M,2022,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,IT1:41_983(1.0),A,001001,KILLINJ,M,2023,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,IT1:41_983(1.0),A,001001,KILLINJ,M,2024,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 113856 entries, 0 to 113855
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          113856 non-null  str    
 1   FREQ              113856 non-null  str    
 2   REF_AREA          113856 non-null  string 
 3   DATA_TYPE         113856 non-null  str    
 4   RESULT            113856 non-null  str    
 5   TIME_PERIOD       113856 non-null  int64  
 6   OBS_VALUE         113856 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(2), str(4)

In [7]:
df_raw.isna().sum()

DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          113856
NOTE_DS             113856
NOTE_REF_AREA       113856
NOTE_DATA_TYPE      113856
NOTE_RESULT         113856
NOTE_TIME_PERIOD    113856
BASE_PER            113856
UNIT_MEAS           113856
UNIT_MULT           113856
dtype: int64

In [8]:
df_traffic = df_raw.dropna(axis=1, how="all").copy()

df_traffic.shape

(113856, 7)

In [9]:
df_traffic.columns.tolist()

['DATAFLOW',
 'FREQ',
 'REF_AREA',
 'DATA_TYPE',
 'RESULT',
 'TIME_PERIOD',
 'OBS_VALUE']

In [10]:
df_traffic.duplicated().sum()

np.int64(0)

In [11]:
df_traffic[["DATA_TYPE", "RESULT"]].drop_duplicates().sort_values(
    ["DATA_TYPE", "RESULT"]
)

,DATA_TYPE,RESULT
0,KILLINJ,F
5,KILLINJ,M
10,ROADACC,9


In [12]:
df_traffic.groupby("TIME_PERIOD")["REF_AREA"].nunique()

TIME_PERIOD
2020    7904
2021    7904
2022    7904
2023    7901
2024    6339
Name: REF_AREA, dtype: int64

In [13]:
df_traffic.groupby(
    ["TIME_PERIOD", "REF_AREA"]
).size().value_counts().sort_index()

3    37952
Name: count, dtype: int64

In [14]:
df_traffic[["DATAFLOW", "FREQ"]].nunique()

DATAFLOW    1
FREQ        1
dtype: int64

In [15]:
df_traffic = df_traffic.drop(
    columns=["DATAFLOW", "FREQ"]
)

df_traffic.shape

(113856, 5)

In [16]:
df_traffic.columns.tolist()

['REF_AREA', 'DATA_TYPE', 'RESULT', 'TIME_PERIOD', 'OBS_VALUE']

In [17]:
# Reshaping

df_traffic.head(10)

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,001001,KILLINJ,F,2020,4
1,001001,KILLINJ,F,2021,2
2,001001,KILLINJ,F,2022,5
3,001001,KILLINJ,F,2023,6
4,001001,KILLINJ,F,2024,5
5,001001,KILLINJ,M,2020,0
6,001001,KILLINJ,M,2021,0
7,001001,KILLINJ,M,2022,0
8,001001,KILLINJ,M,2023,1
9,001001,KILLINJ,M,2024,0


In [18]:
# 1. Accidents

accidents = df_traffic[
    (df_traffic["DATA_TYPE"] == "ROADACC") &
    (df_traffic["RESULT"] == "9")
][["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].copy()

accidents = accidents.rename(
    columns={"OBS_VALUE": "accidents"}
)

accidents.head(10)

,REF_AREA,TIME_PERIOD,accidents
10,001001,2020,4
11,001001,2021,2
12,001001,2022,3
13,001001,2023,3
14,001001,2024,2
25,001002,2020,8
26,001002,2021,6
27,001002,2022,4
28,001002,2023,7
29,001002,2024,5


In [19]:
accidents.shape

(37952, 3)

In [20]:
# 2. Injured

injured = df_traffic[
    (df_traffic["DATA_TYPE"] == "KILLINJ") &
    (df_traffic["RESULT"] == "F")
][["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].copy()

injured = injured.rename(
    columns={"OBS_VALUE": "injured"}
)

injured.head(10)

,REF_AREA,TIME_PERIOD,injured
0,001001,2020,4
1,001001,2021,2
2,001001,2022,5
3,001001,2023,6
4,001001,2024,5
15,001002,2020,14
16,001002,2021,9
17,001002,2022,5
18,001002,2023,10
19,001002,2024,9


In [21]:
injured.shape

(37952, 3)

In [22]:
# 3. Killed

killed = df_traffic[
    (df_traffic["DATA_TYPE"] == "KILLINJ") &
    (df_traffic["RESULT"] == "M")
][["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].copy()

killed = killed.rename(
    columns={"OBS_VALUE": "killed"}
)

killed.head(10)

,REF_AREA,TIME_PERIOD,killed
5,001001,2020,0
6,001001,2021,0
7,001001,2022,0
8,001001,2023,1
9,001001,2024,0
20,001002,2020,0
21,001002,2021,0
22,001002,2022,0
23,001002,2023,1
24,001002,2024,0


In [23]:
killed.shape

(37952, 3)

In [24]:
# MERGE

df_accidents_all = accidents.merge(
    injured,
    on=["REF_AREA", "TIME_PERIOD"],
    validate="one_to_one"
)

df_accidents_all = df_accidents_all.merge(
    killed,
    on=["REF_AREA", "TIME_PERIOD"],
    validate="one_to_one"
)

df_accidents_all.head(10)

,REF_AREA,TIME_PERIOD,accidents,injured,killed
0,001001,2020,4,4,0
1,001001,2021,2,2,0
2,001001,2022,3,5,0
3,001001,2023,3,6,1
4,001001,2024,2,5,0
5,001002,2020,8,14,0
6,001002,2021,6,9,0
7,001002,2022,4,5,0
8,001002,2023,7,10,1
9,001002,2024,5,9,0


In [25]:
df_accidents_all.shape

(37952, 5)

In [26]:
# Important step

df_accidents_all.groupby("TIME_PERIOD")[
    ["accidents", "injured", "killed"]
].sum()

,accidents,injured,killed
TIME_PERIOD,,,
2020,118298,159248,2395
2021,151875,204728,2875
2022,165889,223475,3159
2023,166525,224634,3039
2024,173364,233853,3030


In [27]:
# Web Scraping 2020

situas_url = (
    "https://situas.istat.it/ShibO2Module/api/Report/Spool/"
    "2020-12-31/74"
)

situas_payload = {
    "orderFields": [],
    "orderDirects": [],
    "pFilterFields": [],
    "pFilterValues": []
}

situas_params_test = {
    "PageNum": 1,
    "PageSize": 10
}

response_situas = requests.post(
    situas_url,
    params=situas_params_test,
    json=situas_payload,
    timeout=60
)

response_situas.raise_for_status()

situas_data = response_situas.json()

print("Total rows:", situas_data["totalRows"])

Total rows: 7903


In [28]:
situas_url = (
    "https://situas.istat.it/ShibO2Module/api/Report/Spool/"
    "2020-12-31/74"
)

situas_params_all = {
    "PageNum": 1,
    "PageSize": situas_data["totalRows"]
}

response_situas_all = requests.post(
    situas_url,
    params=situas_params_all,
    json=situas_payload,
    timeout=120
)

response_situas_all.raise_for_status()

print("Status code:", response_situas_all.status_code)

Status code: 200


In [29]:
situas_2020_data = response_situas_all.json()

len(situas_2020_data["body"])

7903

In [30]:
df_situas_2020 = pd.DataFrame(
    situas_2020_data["body"]
)

df_situas_2020.shape

(7903, 17)

In [31]:
df_situas_2020.to_csv(
    "data/raw/situas_municipalities_2020.csv",
    index=False,
    encoding="utf-8-sig"
)

In [32]:
df_situas_2020.columns.tolist()

['COD_RIP',
 'COD_REG',
 'COD_PROV_STORICO',
 'COD_UTS',
 'PRO_COM_T',
 'PRO_COM',
 'COMUNE',
 'SIGLA_AUTOMOBILISTICA',
 'CC_UTS',
 'CC_REG',
 'POP_LEG',
 'ANNO_CENSIMENTO',
 'AREA_KMQ',
 'ANNO_AREA',
 'POP_RES',
 'ANNO_POP_RES',
 'COMUNE_A']

In [33]:
territory_2020 = df_situas_2020[
    [
        "PRO_COM_T",
        "COMUNE",
        "COD_REG",
        "COD_UTS",
        "SIGLA_AUTOMOBILISTICA",
        "AREA_KMQ",
        "POP_RES",
        "ANNO_POP_RES"
    ]
].copy()

In [34]:
territory_2020 = territory_2020.rename(columns={
    "PRO_COM_T": "REF_AREA",
    "COMUNE": "municipality",
    "COD_REG": "region_code",
    "COD_UTS": "province_code",
    "SIGLA_AUTOMOBILISTICA": "province_abbr",
    "AREA_KMQ": "area_km2",
    "POP_RES": "population",
    "ANNO_POP_RES": "TIME_PERIOD"
})

In [35]:
territory_2020["REF_AREA"] = (
    territory_2020["REF_AREA"]
    .astype("string")
    .str.zfill(6)
)

In [36]:
territory_2020.head()

,REF_AREA,municipality,region_code,province_code,province_abbr,area_km2,population,TIME_PERIOD
0,001001,Agliè,01,201,TO,13.1462,2545,2020
1,001002,Airasca,01,201,TO,15.7393,3633,2020
2,001003,Ala di Stura,01,201,TO,46.3315,459,2020
3,001004,Albiano d'Ivrea,01,201,TO,11.7314,1638,2020
4,001006,Almese,01,201,TO,17.8756,6355,2020


In [37]:
territory_2020.shape

(7903, 8)

In [38]:
# quality control

print("Duplicate REF_AREA:", territory_2020["REF_AREA"].duplicated().sum())
print()
print("Missing values:")
print(territory_2020.isna().sum())

Duplicate REF_AREA: 0

Missing values:
REF_AREA         0
municipality     0
region_code      0
province_code    0
province_abbr    0
area_km2         0
population       0
TIME_PERIOD      0
dtype: int64


In [39]:
#equality check

traffic_codes_2020 = set(
    df_accidents_all.loc[
        df_accidents_all["TIME_PERIOD"] == 2020,
        "REF_AREA"
    ]
)

territory_codes_2020 = set(territory_2020["REF_AREA"])

traffic_not_in_territory = traffic_codes_2020 - territory_codes_2020
territory_not_in_traffic = territory_codes_2020 - traffic_codes_2020

print("Traffic not in SITUAS:", traffic_not_in_territory)
print("SITUAS not in traffic:", territory_not_in_traffic)

Traffic not in SITUAS: {'041032'}
SITUAS not in traffic: set()


In [40]:
df_accidents_all[
    (df_accidents_all["TIME_PERIOD"] == 2020) &
    (df_accidents_all["REF_AREA"] == "041032")
]

,REF_AREA,TIME_PERIOD,accidents,injured,killed
19189,041032,2020,0,0,0


In [41]:
df_accidents_all[
    (df_accidents_all["TIME_PERIOD"] == 2020) &
    (df_accidents_all["REF_AREA"] == "041044")
]

,REF_AREA,TIME_PERIOD,accidents,injured,killed
19221,041044,2020,391,469,7


### Territorial change check

Municipality code `041032` required an additional check because of a territorial change recorded by ISTAT.

The traffic record for 2020 was reviewed separately and showed:
- 0 accidents
- 0 injured
- 0 killed

A comparison was also made with the related municipality code `041044`.

This check confirmed that handling this territorial change does not alter the accident totals in the dataset.

In [42]:
#final dataset

territory_2020["TIME_PERIOD"] = territory_2020["TIME_PERIOD"].astype(int)

In [43]:
accidents_2020 = df_accidents_all[
    df_accidents_all["TIME_PERIOD"] == 2020
].copy()

In [44]:
df_final_2020 = territory_2020.merge(
    accidents_2020,
    on=["REF_AREA", "TIME_PERIOD"],
    how="left",
    validate="one_to_one"
)

df_final_2020.shape

(7903, 11)

In [45]:
df_final_2020.info()

<class 'pandas.DataFrame'>
RangeIndex: 7903 entries, 0 to 7902
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   REF_AREA       7903 non-null   string 
 1   municipality   7903 non-null   str    
 2   region_code    7903 non-null   str    
 3   province_code  7903 non-null   str    
 4   province_abbr  7903 non-null   str    
 5   area_km2       7903 non-null   float64
 6   population     7903 non-null   int64  
 7   TIME_PERIOD    7903 non-null   int64  
 8   accidents      7903 non-null   int64  
 9   injured        7903 non-null   int64  
 10  killed         7903 non-null   int64  
dtypes: float64(1), int64(5), str(4), string(1)
memory usage: 679.3 KB


In [46]:
df_final_2020.head()

,REF_AREA,municipality,region_code,province_code,province_abbr,area_km2,population,TIME_PERIOD,accidents,injured,killed
0,001001,Agliè,01,201,TO,13.1462,2545,2020,4,4,0
1,001002,Airasca,01,201,TO,15.7393,3633,2020,8,14,0
2,001003,Ala di Stura,01,201,TO,46.3315,459,2020,0,0,0
3,001004,Albiano d'Ivrea,01,201,TO,11.7314,1638,2020,3,3,0
4,001006,Almese,01,201,TO,17.8756,6355,2020,6,8,0


In [47]:
df_final_2020[
    ["accidents", "injured", "killed"]
].isna().sum()

accidents    0
injured      0
killed       0
dtype: int64

In [48]:
df_final_2020[
    ["accidents", "injured", "killed"]
].sum()

accidents    118298
injured      159248
killed         2395
dtype: int64

In [49]:
#  Web Scraping 2021

situas_url_2021 = (
    "https://situas.istat.it/ShibO2Module/api/Report/Spool/"
    "2021-12-31/74"
)
situas_params_test = {
    "PageNum": 1,
    "PageSize": 10
}

response_situas_2021 = requests.post(
    situas_url_2021,
    params=situas_params_test,
    json=situas_payload,
    timeout=60
)

response_situas_2021.raise_for_status()

situas_2021_test = response_situas_2021.json()

print("Total rows:", situas_2021_test["totalRows"])

Total rows: 7904


In [50]:
situas_params_all_2021 = {
    "PageNum": 1,
    "PageSize": situas_2021_test["totalRows"]
}

response_situas_2021_all = requests.post(
    situas_url_2021,
    params=situas_params_all_2021,
    json=situas_payload,
    timeout=120
)

response_situas_2021_all.raise_for_status()

situas_2021_data = response_situas_2021_all.json()

len(situas_2021_data["body"])

7904

In [51]:
df_situas_2021 = pd.DataFrame(
    situas_2021_data["body"]
)

df_situas_2021.shape

(7904, 17)

In [52]:
df_situas_2021.to_csv(
    "data/raw/situas_municipalities_2021.csv",
    index=False,
    encoding="utf-8-sig"
)

In [53]:
territory_2021 = df_situas_2021[
    [
        "PRO_COM_T",
        "COMUNE",
        "COD_REG",
        "COD_UTS",
        "SIGLA_AUTOMOBILISTICA",
        "AREA_KMQ",
        "POP_RES",
        "ANNO_POP_RES"
    ]
].copy()

In [54]:
territory_2021 = territory_2021.rename(columns={
    "PRO_COM_T": "REF_AREA",
    "COMUNE": "municipality",
    "COD_REG": "region_code",
    "COD_UTS": "province_code",
    "SIGLA_AUTOMOBILISTICA": "province_abbr",
    "AREA_KMQ": "area_km2",
    "POP_RES": "population",
    "ANNO_POP_RES": "TIME_PERIOD"
})

In [55]:
territory_2021["REF_AREA"] = (
    territory_2021["REF_AREA"]
    .astype("string")
    .str.zfill(6)
)

territory_2021["TIME_PERIOD"] = territory_2021["TIME_PERIOD"].astype(int)

In [56]:
territory_2021.head()

,REF_AREA,municipality,region_code,province_code,province_abbr,area_km2,population,TIME_PERIOD
0,001001,Agliè,01,201,TO,13.1463,2562,2021
1,001002,Airasca,01,201,TO,15.7393,3660,2021
2,001003,Ala di Stura,01,201,TO,46.3316,467,2021
3,001004,Albiano d'Ivrea,01,201,TO,11.7397,1637,2021
4,001006,Almese,01,201,TO,17.8741,6331,2021


In [57]:
territory_2021.shape

(7904, 8)

In [58]:
# quality control

print("Duplicate REF_AREA:", territory_2021["REF_AREA"].duplicated().sum())
print()
print("Missing values:")
print(territory_2021.isna().sum())

Duplicate REF_AREA: 0

Missing values:
REF_AREA         0
municipality     0
region_code      0
province_code    0
province_abbr    0
area_km2         0
population       0
TIME_PERIOD      0
dtype: int64


In [59]:
#equality check

traffic_codes_2021 = set(
    df_accidents_all.loc[
        df_accidents_all["TIME_PERIOD"] == 2021,
        "REF_AREA"
    ]
)

territory_codes_2021 = set(territory_2021["REF_AREA"])

traffic_codes_2021 == territory_codes_2021

True

In [60]:
#final dataset

accidents_2021 = df_accidents_all[
    df_accidents_all["TIME_PERIOD"] == 2021
].copy()

In [61]:
df_final_2021 = territory_2021.merge(
    accidents_2021,
    on=["REF_AREA", "TIME_PERIOD"],
    how="left",
    validate="one_to_one"
)

df_final_2021.shape

(7904, 11)

In [62]:
df_final_2021[
    ["accidents", "injured", "killed"]
].isna().sum()

accidents    0
injured      0
killed       0
dtype: int64

In [63]:
df_final_2021[
    ["accidents", "injured", "killed"]
].sum()

accidents    151875
injured      204728
killed         2875
dtype: int64

In [64]:
#  Web Scraping 2022

situas_url_2022 = (
    "https://situas.istat.it/ShibO2Module/api/Report/Spool/"
    "2022-12-31/74"
)

response_situas_2022 = requests.post(
    situas_url_2022,
    params={
        "PageNum": 1,
        "PageSize": 10
    },
    json=situas_payload,
    timeout=60
)

response_situas_2022.raise_for_status()

situas_2022_test = response_situas_2022.json()

print("Total rows:", situas_2022_test["totalRows"])

Total rows: 7904


In [65]:
situas_params_all_2022 = {
    "PageNum": 1,
    "PageSize": situas_2022_test["totalRows"]
}

response_situas_2022_all = requests.post(
    situas_url_2022,
    params=situas_params_all_2022,
    json=situas_payload,
    timeout=120
)

response_situas_2022_all.raise_for_status()

situas_2022_data = response_situas_2022_all.json()

len(situas_2022_data["body"])

7904

In [66]:
df_situas_2022 = pd.DataFrame(
    situas_2022_data["body"]
)
df_situas_2022.shape

(7904, 17)

In [67]:
df_situas_2022.to_csv(
    "data/raw/situas_municipalities_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

In [68]:
territory_2022 = df_situas_2022[
    [
        "PRO_COM_T",
        "COMUNE",
        "COD_REG",
        "COD_UTS",
        "SIGLA_AUTOMOBILISTICA",
        "AREA_KMQ",
        "POP_RES",
        "ANNO_POP_RES"
    ]
].copy()

territory_2022 = territory_2022.rename(columns={
    "PRO_COM_T": "REF_AREA",
    "COMUNE": "municipality",
    "COD_REG": "region_code",
    "COD_UTS": "province_code",
    "SIGLA_AUTOMOBILISTICA": "province_abbr",
    "AREA_KMQ": "area_km2",
    "POP_RES": "population",
    "ANNO_POP_RES": "TIME_PERIOD"
})

territory_2022["REF_AREA"] = (
    territory_2022["REF_AREA"]
    .astype("string")
    .str.zfill(6)
)

territory_2022["TIME_PERIOD"] = territory_2022["TIME_PERIOD"].astype(int)

territory_2022.shape

(7904, 8)

In [69]:
# quality control

print("Duplicate REF_AREA:", territory_2022["REF_AREA"].duplicated().sum())
print()
print("Missing values:")
print(territory_2022.isna().sum())

Duplicate REF_AREA: 0

Missing values:
REF_AREA         0
municipality     0
region_code      0
province_code    0
province_abbr    0
area_km2         0
population       0
TIME_PERIOD      0
dtype: int64


In [70]:
#equality check

traffic_codes_2022 = set(
    df_accidents_all.loc[
        df_accidents_all["TIME_PERIOD"] == 2022,
        "REF_AREA"
    ]
)

territory_codes_2022 = set(territory_2022["REF_AREA"])

traffic_codes_2022 == territory_codes_2022

True

In [71]:
#final dataset

accidents_2022 = df_accidents_all[
    df_accidents_all["TIME_PERIOD"] == 2022
].copy()

df_final_2022 = territory_2022.merge(
    accidents_2022,
    on=["REF_AREA", "TIME_PERIOD"],
    how="left",
    validate="one_to_one"
)

df_final_2022.shape

(7904, 11)

In [72]:
df_final_2022[
    ["accidents", "injured", "killed"]
].sum()

accidents    165889
injured      223475
killed         3159
dtype: int64

In [73]:
#  Web Scraping 2023

situas_url_2023 = (
    "https://situas.istat.it/ShibO2Module/api/Report/Spool/"
    "2023-12-31/74"
)

response_situas_2023 = requests.post(
    situas_url_2023,
    params={
        "PageNum": 1,
        "PageSize": 10
    },
    json=situas_payload,
    timeout=60
)

response_situas_2023.raise_for_status()

situas_2023_test = response_situas_2023.json()

print("Total rows:", situas_2023_test["totalRows"])

Total rows: 7900


In [74]:
situas_params_all_2023 = {
    "PageNum": 1,
    "PageSize": situas_2023_test["totalRows"]
}

response_situas_2023_all = requests.post(
    situas_url_2023,
    params=situas_params_all_2023,
    json=situas_payload,
    timeout=120
)

response_situas_2023_all.raise_for_status()

situas_2023_data = response_situas_2023_all.json()

len(situas_2023_data["body"])

7900

In [75]:
df_situas_2023 = pd.DataFrame(
    situas_2023_data["body"]
)

df_situas_2023.shape

(7900, 17)

In [76]:
df_situas_2023.to_csv(
    "data/raw/situas_municipalities_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

In [77]:
territory_2023 = df_situas_2023[
    [
        "PRO_COM_T",
        "COMUNE",
        "COD_REG",
        "COD_UTS",
        "SIGLA_AUTOMOBILISTICA",
        "AREA_KMQ",
        "POP_RES",
        "ANNO_POP_RES"
    ]
].copy()

territory_2023 = territory_2023.rename(columns={
    "PRO_COM_T": "REF_AREA",
    "COMUNE": "municipality",
    "COD_REG": "region_code",
    "COD_UTS": "province_code",
    "SIGLA_AUTOMOBILISTICA": "province_abbr",
    "AREA_KMQ": "area_km2",
    "POP_RES": "population",
    "ANNO_POP_RES": "TIME_PERIOD"
})

territory_2023["REF_AREA"] = (
    territory_2023["REF_AREA"]
    .astype("string")
    .str.zfill(6)
)

territory_2023["TIME_PERIOD"] = territory_2023["TIME_PERIOD"].astype(int)

territory_2023.shape

(7900, 8)

In [78]:
#equality check

traffic_codes_2023 = set(
    df_accidents_all.loc[
        df_accidents_all["TIME_PERIOD"] == 2023,
        "REF_AREA"
    ]
)

territory_codes_2023 = set(territory_2023["REF_AREA"])

traffic_codes_2023 == territory_codes_2023

False

In [79]:
traffic_codes_2023 - territory_codes_2023

{'018002'}

In [80]:
df_accidents_all[
    (df_accidents_all["TIME_PERIOD"] == 2023) &
    (df_accidents_all["REF_AREA"] == "018002")
]

,REF_AREA,TIME_PERIOD,accidents,injured,killed
10868,018002,2023,0,0,0


In [81]:
# quality control

print("Duplicate REF_AREA:", territory_2023["REF_AREA"].duplicated().sum())
print()
print("Missing values:")
print(territory_2023.isna().sum())

Duplicate REF_AREA: 0

Missing values:
REF_AREA         0
municipality     0
region_code      0
province_code    0
province_abbr    0
area_km2         0
population       0
TIME_PERIOD      0
dtype: int64


In [82]:
#final dataset

accidents_2023 = df_accidents_all[
    df_accidents_all["TIME_PERIOD"] == 2023
].copy()

In [83]:
df_final_2023 = territory_2023.merge(
    accidents_2023,
    on=["REF_AREA", "TIME_PERIOD"],
    how="left",
    validate="one_to_one"
)

df_final_2023.shape

(7900, 11)

In [84]:
df_final_2023[
    ["accidents", "injured", "killed"]
].sum()

accidents    166525
injured      224634
killed         3039
dtype: int64

In [85]:
#  Web Scraping 2024

situas_url_2024 = (
    "https://situas.istat.it/ShibO2Module/api/Report/Spool/"
    "2024-12-31/74"
)

response_situas_2024 = requests.post(
    situas_url_2024,
    params={
        "PageNum": 1,
        "PageSize": 10
    },
    json=situas_payload,
    timeout=60
)

response_situas_2024.raise_for_status()

situas_2024_test = response_situas_2024.json()

print("Total rows:", situas_2024_test["totalRows"])

Total rows: 7896


In [86]:
situas_params_all_2024 = {
    "PageNum": 1,
    "PageSize": situas_2024_test["totalRows"]
}

response_situas_2024_all = requests.post(
    situas_url_2024,
    params=situas_params_all_2024,
    json=situas_payload,
    timeout=120
)

response_situas_2024_all.raise_for_status()

situas_2024_data = response_situas_2024_all.json()

len(situas_2024_data["body"])

7896

In [87]:
df_situas_2024 = pd.DataFrame(
    situas_2024_data["body"]
)

df_situas_2024.shape

(7896, 17)

In [88]:
df_situas_2024.to_csv(
    "data/raw/situas_municipalities_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

In [89]:
territory_2024 = df_situas_2024[
    [
        "PRO_COM_T",
        "COMUNE",
        "COD_REG",
        "COD_UTS",
        "SIGLA_AUTOMOBILISTICA",
        "AREA_KMQ",
        "POP_RES",
        "ANNO_POP_RES"
    ]
].copy()

territory_2024 = territory_2024.rename(columns={
    "PRO_COM_T": "REF_AREA",
    "COMUNE": "municipality",
    "COD_REG": "region_code",
    "COD_UTS": "province_code",
    "SIGLA_AUTOMOBILISTICA": "province_abbr",
    "AREA_KMQ": "area_km2",
    "POP_RES": "population",
    "ANNO_POP_RES": "TIME_PERIOD"
})

territory_2024["REF_AREA"] = (
    territory_2024["REF_AREA"]
    .astype("string")
    .str.zfill(6)
)

territory_2024["TIME_PERIOD"] = territory_2024["TIME_PERIOD"].astype(int)

territory_2024.shape

(7896, 8)

In [90]:
#equality check

print("Duplicate REF_AREA:", territory_2024["REF_AREA"].duplicated().sum())
print()
print("Missing values:")
print(territory_2024.isna().sum())

Duplicate REF_AREA: 0

Missing values:
REF_AREA         0
municipality     0
region_code      0
province_code    0
province_abbr    0
area_km2         0
population       0
TIME_PERIOD      0
dtype: int64


In [91]:
traffic_codes_2024 = set(
    df_accidents_all.loc[
        df_accidents_all["TIME_PERIOD"] == 2024,
        "REF_AREA"
    ]
)

territory_codes_2024 = set(territory_2024["REF_AREA"])

traffic_codes_2024.issubset(territory_codes_2024)

True

In [92]:
accidents_2024 = df_accidents_all[
    df_accidents_all["TIME_PERIOD"] == 2024
].copy()

In [93]:
df_final_2024 = territory_2024.merge(
    accidents_2024,
    on=["REF_AREA", "TIME_PERIOD"],
    how="left",
    validate="one_to_one"
)

df_final_2024.shape

(7896, 11)

In [94]:
df_final_2024[
    ["accidents", "injured", "killed"]
].isna().sum()

accidents    1557
injured      1557
killed       1557
dtype: int64

### Handling municipalities without traffic records in 2024

After merging the 2024 SITUAS municipality data with the road accident data,
1,557 municipalities had no corresponding traffic record.

For this analysis, the absence of a traffic record is interpreted as no
recorded accidents, injured people or fatalities for that municipality in 2024.

Therefore, the missing traffic values are replaced with zero.

The totals are checked afterwards to confirm that this operation does not
change the original 2024 traffic totals.

In [95]:
df_final_2024[
    ["accidents", "injured", "killed"]
] = df_final_2024[
    ["accidents", "injured", "killed"]
].fillna(0).astype(int)

In [96]:
df_final_2024[
    ["accidents", "injured", "killed"]
].isna().sum()

accidents    0
injured      0
killed       0
dtype: int64

In [97]:
df_final_2024[
    ["accidents", "injured", "killed"]
].sum()

accidents    173364
injured      233853
killed         3030
dtype: int64

## Merge the five final 2020–2024 datasets into a single DataFrame.

In [98]:
print(df_final_2020.columns.tolist())
print(df_final_2021.columns.tolist())
print(df_final_2022.columns.tolist())
print(df_final_2023.columns.tolist())
print(df_final_2024.columns.tolist())

['REF_AREA', 'municipality', 'region_code', 'province_code', 'province_abbr', 'area_km2', 'population', 'TIME_PERIOD', 'accidents', 'injured', 'killed']
['REF_AREA', 'municipality', 'region_code', 'province_code', 'province_abbr', 'area_km2', 'population', 'TIME_PERIOD', 'accidents', 'injured', 'killed']
['REF_AREA', 'municipality', 'region_code', 'province_code', 'province_abbr', 'area_km2', 'population', 'TIME_PERIOD', 'accidents', 'injured', 'killed']
['REF_AREA', 'municipality', 'region_code', 'province_code', 'province_abbr', 'area_km2', 'population', 'TIME_PERIOD', 'accidents', 'injured', 'killed']
['REF_AREA', 'municipality', 'region_code', 'province_code', 'province_abbr', 'area_km2', 'population', 'TIME_PERIOD', 'accidents', 'injured', 'killed']


In [99]:
df_final_all = pd.concat(
    [
        df_final_2020,
        df_final_2021,
        df_final_2022,
        df_final_2023,
        df_final_2024
    ],
    ignore_index=True
)

df_final_all.shape

(39507, 11)

In [100]:
df_final_all["TIME_PERIOD"].value_counts().sort_index()

TIME_PERIOD
2020    7903
2021    7904
2022    7904
2023    7900
2024    7896
Name: count, dtype: int64

In [101]:
# Let's check whether population and area_km2 can be used as denominators.

print("Population <= 0:", (df_final_all["population"] <= 0).sum())
print("Area <= 0:", (df_final_all["area_km2"] <= 0).sum())

print()
print("Missing population:", df_final_all["population"].isna().sum())
print("Missing area:", df_final_all["area_km2"].isna().sum())

Population <= 0: 0
Area <= 0: 0

Missing population: 0
Missing area: 0


Indicators based on the complete 2020–2024 dataset: 
- "accidents_per_100k"
- "accidents_per_km2"

In [102]:
df_final_all["accidents_per_100k"] = (
    df_final_all["accidents"] / df_final_all["population"]
) * 100000

In [103]:
df_final_all["accidents_per_km2"] = (
    df_final_all["accidents"] / df_final_all["area_km2"]
)

In [104]:
df_final_all[
    [
        "REF_AREA",
        "municipality",
        "TIME_PERIOD",
        "accidents",
        "population",
        "area_km2",
        "accidents_per_100k",
        "accidents_per_km2"
    ]
].head()

,REF_AREA,municipality,TIME_PERIOD,accidents,population,area_km2,accidents_per_100k,accidents_per_km2
0,001001,Agliè,2020,4,2545,13.1462,157.170923,0.304270
1,001002,Airasca,2020,8,3633,15.7393,220.203688,0.508282
2,001003,Ala di Stura,2020,0,459,46.3315,0.000000,0.000000
3,001004,Albiano d'Ivrea,2020,3,1638,11.7314,183.150183,0.255724
4,001006,Almese,2020,6,6355,17.8756,94.413847,0.335653


In [105]:
#equality check

print("Shape:", df_final_all.shape)

print(
    "Duplicate municipality-year:",
    df_final_all.duplicated(
        subset=["REF_AREA", "TIME_PERIOD"]
    ).sum()
)

print()
print("Missing values:")
print(df_final_all.isna().sum())

Shape: (39507, 13)
Duplicate municipality-year: 0

Missing values:
REF_AREA              0
municipality          0
region_code           0
province_code         0
province_abbr         0
area_km2              0
population            0
TIME_PERIOD           0
accidents             0
injured               0
killed                0
accidents_per_100k    0
accidents_per_km2     0
dtype: int64


In [106]:
df_final_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 39507 entries, 0 to 39506
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   REF_AREA            39507 non-null  string 
 1   municipality        39507 non-null  str    
 2   region_code         39507 non-null  str    
 3   province_code       39507 non-null  str    
 4   province_abbr       39507 non-null  str    
 5   area_km2            39507 non-null  float64
 6   population          39507 non-null  int64  
 7   TIME_PERIOD         39507 non-null  int64  
 8   accidents           39507 non-null  int64  
 9   injured             39507 non-null  int64  
 10  killed              39507 non-null  int64  
 11  accidents_per_100k  39507 non-null  float64
 12  accidents_per_km2   39507 non-null  float64
dtypes: float64(3), int64(5), str(4), string(1)
memory usage: 3.9 MB


Final cleaned dataset in data/cleaned:

In [107]:
cleaned_path = Path(
    "data/cleaned/istat_traffic_2020_2024_cleaned.csv"
)

df_final_all.to_csv(
    cleaned_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:", cleaned_path)

Saved to: data\cleaned\istat_traffic_2020_2024_cleaned.csv
